# 4 Data Exploration

This notebook is a Renku-friendly demo for exploring the published MOTEL data product.

It stays focused on operations and usage:
- the notebook shows the user workflow
- `config.py` stores paths and default demo parameters
- `exploration_utils.py` stores reusable data loading and analysis helpers


## Run Order

Run the notebook from top to bottom.

For a Renku demo, a good flow is:
1. load and inspect the prepared tables
2. run one use case with the default parameters
3. change one parameter live and rerun the relevant section

Instruction:
- Run every cell in sequence the first time.
- After that, you usually only need to rerun one input cell and the result cell directly below it.


## Step 1: Load The Data And Helper Functions

What this step does:
- imports plotting and table display tools
- imports project defaults from `config.py`
- imports reusable exploration logic from `exploration_utils.py`
- loads the canonical `linked_entity` dataset and converts it into analysis-ready tables

Instruction:
- Run the next cell once.
- If it fails, check that the notebook is being run from the `4_data_explore` folder and that the project files are present.
- The output should be a single number showing how many linked entities were loaded.


In [10]:
## --- RENKU USER ---
# to see the plotlyi plots correct on Renku, you need to set the default renderer to "iframe"
# just uncomment the following line and run the script again

# import plotly.io as pio
# pio.renderers.default = "iframe"

In [11]:
import pandas as pd
import plotly.express as px
from IPython.display import display

from config import (
    DEFAULT_ATTRIBUTE_OF_INTEREST,
    DEFAULT_CARRIER_QUERY,
    DEFAULT_SOURCE_KEYWORDS,
    DEFAULT_TECHNOLOGY_KEYWORDS,
    LINKED_ENTITY_PATH,
    MAPPING_DIR,
    VOCAB_DIR,
)
from exploration_utils import (
    build_overview_df,
    load_exploration_context,
    prepare_carrier_analysis,
    prepare_source_analysis,
    prepare_technology_analysis,
    preview_entities,
)

# Load the canonical linked-entity data and the flat tables used below.
context = load_exploration_context(LINKED_ENTITY_PATH, MAPPING_DIR, VOCAB_DIR)

linked_entities = context["linked_entities"]
entities_df = context["entities_df"]
values_df = context["values_df"]
sources_df = context["sources_df"]
source_attributes_df = context["source_attributes_df"]
carriers_df = context["carriers_df"]

len(linked_entities)

186

## Step 2: Inspect The Loaded Tables

What this step does:
- gives a compact summary of the dataset
- shows a preview of the main entity table
- helps you confirm that the load step worked before moving into the use cases

Instruction:
- Run the next cell.
- Read the overview table first.
- Then scan the entity preview to see the columns that will be reused later.
- If the preview looks wrong, stop here and debug before continuing.


In [12]:
# Build a high-level summary for a quick dataset health check.
overview_df = build_overview_df(entities_df, values_df, sources_df, carriers_df)
display(overview_df)

# Preview the main exploration table used in the use cases below.
preview_entities(entities_df)

,metric,value
0,linked entities,186
1,technologies,66
2,processes,55
3,sources,31
4,attributes in values table,28
5,carriers,33
6,reference years,"2025, 2030, 2040, 2050"


,linked_entity_id,tech_id,tech_name,process_id,process_name,reference_year,input_carriers,output_carriers,source_count,value_count
0,LE_00001,TECH_00001,NH3 CCGT El,PROC_00001,Methane-Fueled CCGT with CCS Electricity Gener...,2050,Ammonia,Grid Electricity Level 1-3,3,11
1,LE_00002,TECH_00002,AmmoniaOCGT,PROC_00002,Methane-Fueled Open Cycle Gas Turbine Electric...,2050,Ammonia,Grid Electricity Level 1-3,3,11
2,LE_00003,TECH_00003,CH4 CCGT CCS,PROC_00001,Methane-Fueled CCGT with CCS Electricity Gener...,2050,Methane,Grid Electricity Level 1-3,3,14
3,LE_00004,TECH_00004,Methane CHP,PROC_00003,Combined Heat and Power (CHP) Generation using...,2050,Methane,"Grid Electricity, High-Temperature Heat",3,13
4,LE_00005,TECH_00005,CH4 Fuel Cell El,PROC_00004,Methane-Fueled Fuel Cell Electricity and Heat ...,2030,Methane,"Grid Electricity, High-Temperature Heat",2,13
5,LE_00006,TECH_00005,CH4 Fuel Cell El,PROC_00004,Methane-Fueled Fuel Cell Electricity and Heat ...,2030,Methane,"Electricity in Buildings, Thermal Energy in Bu...",2,13
6,LE_00007,TECH_00006,CH4 OCGT Electricity,PROC_00002,Methane-Fueled Open Cycle Gas Turbine Electric...,2050,Methane,Grid Electricity Level 1-3,3,14
7,LE_00008,TECH_00007,Geothermal CHP Electricity and Heat,PROC_00005,Geothermal CHP Generation,2050,Geothermal Heat,"Grid Electricity, High-Temperature Heat",3,12
8,LE_00009,TECH_00008,H2 CCGT Electricity,PROC_00001,Methane-Fueled CCGT with CCS Electricity Gener...,2050,Hydrogen,Grid Electricity Level 1-3,4,14
9,LE_00010,TECH_00009,El Fuel Cell SOFC,PROC_00004,Methane-Fueled Fuel Cell Electricity and Heat ...,2040,Hydrogen,"Grid Electricity, High-Temperature Heat",3,13


## How To Use The Demo Inputs

The next sections expose only a few parameters.

Instruction:
- Edit values only in the small input cells.
- Leave the result cells unchanged unless you are extending the workflow.
- When you change an input, rerun that input cell and then the result cell below it.

This keeps the notebook easy to operate live while the heavier logic stays in reusable Python files.

## Use Case 1: Search By Technology Or Process

Typical user question:

> Which hydrogen or fuel-cell related technologies are available, which sources support them, and how do their capital costs compare across years?

This use case is good for a demo when the audience starts from a technology family rather than from a source or a carrier.

### Step 3A: Choose Technology Keywords And One Attribute

Instruction:
- Edit `technology_keywords` if you want a different topic, for example `['solar']`, `['battery']`, or `['methanol']`.
- Edit `attribute_of_interest` if you want a different numeric comparison, for example `Technical Efficiency` or `Economic Lifetime`.
- Run the next cell after changing either input.


In [13]:
# Set the technology or process terms you want to search for.
technology_keywords = DEFAULT_TECHNOLOGY_KEYWORDS

# Choose the attribute that should be compared across the selected technologies.
attribute_of_interest = DEFAULT_ATTRIBUTE_OF_INTEREST

display(pd.DataFrame({"technology_keywords": technology_keywords}))
attribute_of_interest

,technology_keywords
0,hydrogen
1,fuel cell


'Capital Expenditure Per Capacity'

### Step 3B: Run The Technology Search And Read The Outputs

What this step does:
- filters the entity table using your keywords
- shows the matching entities first
- summarizes which sources support those entities
- summarizes which attributes appear most often
- plots the selected numeric attribute across years

Instruction:
- Run the next cell.
- Read the outputs from top to bottom.
- For a live demo, start with the table, then the source bar chart, then the final trend line.


In [14]:
# Prepare the filtered tables used across the displays and charts below.
technology_analysis = prepare_technology_analysis(
    entities_df,
    values_df,
    sources_df,
    technology_keywords,
    attribute_of_interest,
)

selected_entities = technology_analysis["selected_entities"]
selected_source_summary = technology_analysis["selected_source_summary"]
selected_attribute_summary = technology_analysis["selected_attribute_summary"]
selected_attribute_values = technology_analysis["selected_attribute_values"]

display(
    preview_entities(selected_entities, limit=len(selected_entities))
)

px.bar(
    selected_source_summary,
    x="source_name",
    y="linked_attributes",
    color="linked_entities",
    title="Source coverage for the selected technologies",
    labels={"source_name": "Source", "linked_attributes": "Linked attributes"},
).show()

px.bar(
    selected_attribute_summary,
    x="records",
    y="attribute_name",
    orientation="h",
    title="Most common attributes across the selected technologies",
).show()

# Show the exact rows behind the final comparison chart.
display(
    selected_attribute_values[
        [
            "linked_entity_id",
            "tech_name",
            "process_name",
            "reference_year",
            "attribute_name",
            "value",
            "time_index",
        ]
    ]
)

px.line(
    selected_attribute_values,
    x="reference_year",
    y="value_numeric",
    color="tech_name",
    markers=True,
    hover_data=["process_name", "linked_entity_id", "time_index"],
    title=f"{attribute_of_interest} by reference year",
    labels={"reference_year": "Reference year", "value_numeric": attribute_of_interest},
).show()

,linked_entity_id,tech_id,tech_name,process_id,process_name,reference_year,input_carriers,output_carriers,source_count,value_count
4,LE_00005,TECH_00005,CH4 Fuel Cell El,PROC_00004,Methane-Fueled Fuel Cell Electricity and Heat ...,2030,Methane,"Grid Electricity, High-Temperature Heat",2,13
5,LE_00006,TECH_00005,CH4 Fuel Cell El,PROC_00004,Methane-Fueled Fuel Cell Electricity and Heat ...,2030,Methane,"Electricity in Buildings, Thermal Energy in Bu...",2,13
129,LE_00130,TECH_00005,CH4 Fuel Cell El,PROC_00004,Methane-Fueled Fuel Cell Electricity and Heat ...,<NA>,,,0,8
130,LE_00131,TECH_00005,CH4 Fuel Cell El,PROC_00004,Methane-Fueled Fuel Cell Electricity and Heat ...,<NA>,,,0,8
57,LE_00058,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2025,"Carbon Dioxide, Hydrogen","High-Temperature Heat, Methanol",3,12
58,LE_00059,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2030,"Carbon Dioxide, Hydrogen","High-Temperature Heat, Methanol",3,12
59,LE_00060,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2040,"Carbon Dioxide, Hydrogen","High-Temperature Heat, Methanol",4,12
60,LE_00061,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2040,"Carbon Monoxide, Hydrogen","High-Temperature Heat, Methanol",3,12
61,LE_00062,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2050,"Carbon Dioxide, Hydrogen","High-Temperature Heat, Methanol",3,12
149,LE_00150,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,<NA>,,,0,8


,linked_entity_id,tech_name,process_name,reference_year,attribute_name,value,time_index
56,LE_00005,CH4 Fuel Cell El,Methane-Fueled Fuel Cell Electricity and Heat ...,2030,Capital Expenditure Per Capacity,6000,2030
69,LE_00006,CH4 Fuel Cell El,Methane-Fueled Fuel Cell Electricity and Heat ...,2030,Capital Expenditure Per Capacity,10000,2030
715,LE_00058,CO2 Hydrogenation to Methanol,Methanol Synthesis,2025,Capital Expenditure Per Capacity,810,2025
727,LE_00059,CO2 Hydrogenation to Methanol,Methanol Synthesis,2030,Capital Expenditure Per Capacity,750,2030
739,LE_00060,CO2 Hydrogenation to Methanol,Methanol Synthesis,2040,Capital Expenditure Per Capacity,793,2040
751,LE_00061,CO2 Hydrogenation to Methanol,Methanol Synthesis,2040,Capital Expenditure Per Capacity,700,2040
763,LE_00062,CO2 Hydrogenation to Methanol,Methanol Synthesis,2050,Capital Expenditure Per Capacity,630,2050
787,LE_00064,El Electrolysis Alkaline,Hydrogen Production via Alkaline Electrolysis,2030,Capital Expenditure Per Capacity,875,2030
800,LE_00065,El Electrolysis Alkaline,Hydrogen Production via Alkaline Electrolysis,2040,Capital Expenditure Per Capacity,675,2040
813,LE_00066,El Electrolysis Alkaline,Hydrogen Production via Alkaline Electrolysis,2050,Capital Expenditure Per Capacity,475,2050


## Use Case 2: Search By Source

Typical user question:

> Which sources contribute the most linked attributes, what kinds of attributes do they contribute, and how wide is their technology coverage?

This use case is helpful when users want to understand provenance and dataset coverage.

### Step 4A: Choose One Or More Source Keywords

Instruction:
- Edit `source_keywords` to focus on one source family or compare several.
- Good demo examples are short fragments such as `['VSE']`, `['ShareRef']`, or `['DanishEnergyAgency']`.
- Run the next cell after updating the list.


In [15]:
# Set the source names or source-name fragments you want to inspect.
source_keywords = DEFAULT_SOURCE_KEYWORDS
display(pd.DataFrame({"source_keywords": source_keywords}))

,source_keywords
0,VSE
1,ShareRef
2,DanishEnergyAgency


### Step 4B: Run The Source Analysis

What this step does:
- filters the source-to-attribute table by your chosen source keywords
- summarizes how many linked attributes each source contributes
- shows how broad each source is across technologies and processes
- shows the most common attribute types contributed by each source

Instruction:
- Run the next cell.
- Use the summary table to orient yourself first.
- Then use the three charts to explain contribution volume, breadth, and attribute mix.


In [16]:
# Prepare the source slice used in the summary table and charts below.
source_analysis = prepare_source_analysis(source_attributes_df, source_keywords)

selected_source_attributes = source_analysis["selected_source_attributes"]
source_attribute_summary = source_analysis["source_attribute_summary"]
top_attribute_types = source_analysis["top_attribute_types"]

display(source_attribute_summary)

px.bar(
    source_attribute_summary,
    x="source_name",
    y="linked_attributes",
    color="distinct_attribute_types",
    title="How many linked attributes each source contributes",
    labels={"source_name": "Source", "linked_attributes": "Linked attributes"},
).show()

px.bar(
    source_attribute_summary,
    x="source_name",
    y="distinct_technologies",
    color="distinct_processes",
    title="Technology coverage breadth by source",
    labels={"source_name": "Source", "distinct_technologies": "Distinct technologies"},
).show()

px.bar(
    top_attribute_types,
    x="records",
    y="attribute_name",
    color="source_name",
    facet_col="source_name",
    facet_col_wrap=2,
    orientation="h",
    title="Top attribute types proposed by each selected source",
).show()

,source_name,linked_attributes,distinct_attribute_types,distinct_technologies,distinct_processes
3,VSE2025,147,5,26,22
1,ShareRef,96,2,32,25
2,VSE2022,66,8,29,26
0,DanishEnergyAgency2025,50,7,8,7


## Use Case 3: Search By Energy Carrier

Typical user question:

> If I start from a carrier such as hydrogen, which technologies consume it or produce it, and in which years do those entities appear?

This use case is useful when the audience thinks in terms of inputs and outputs rather than technologies.

### Step 5A: Choose A Carrier Query

Instruction:
- Edit `carrier_query` to a carrier name or fragment such as `hydrogen`, `electricity`, `methanol`, or `ammonia`.
- Run the next cell after changing the query.


In [17]:
# Set the carrier name or partial carrier name you want to explore.
carrier_query = DEFAULT_CARRIER_QUERY
carrier_query

'hydrogen'

### Step 5B: Run The Carrier Analysis

What this step does:
- filters the carrier table using your carrier query
- shows all matching carrier rows first
- groups them into a simpler technology-level summary
- plots how many linked entities are connected to the carrier on the input or output side

Instruction:
- Run the next cell.
- Use the first table to show the raw evidence.
- Use the second table and chart to present the simplified story.


In [18]:
# Prepare the carrier slice used in the detailed table and chart below.
carrier_analysis = prepare_carrier_analysis(carriers_df, carrier_query)

matched_carriers = carrier_analysis["matched_carriers"]
technology_by_carrier = carrier_analysis["technology_by_carrier"]

display(
    matched_carriers[
        [
            "linked_entity_id",
            "tech_id",
            "tech_name",
            "process_id",
            "process_name",
            "reference_year",
            "direction",
            "carrier_id",
            "carrier_name",
            "carrier_type",
            "share",
            "unit",
        ]
    ].sort_values(["direction", "tech_name", "reference_year"])
)

display(technology_by_carrier)

px.bar(
    technology_by_carrier,
    x="tech_name",
    y="linked_entities",
    color="direction",
    hover_data=["process_name", "first_year", "last_year"],
    title=f"Technologies connected to carrier query: {carrier_query}",
    labels={"tech_name": "Technology", "linked_entities": "Linked entities"},
).show()

,linked_entity_id,tech_id,tech_name,process_id,process_name,reference_year,direction,carrier_id,carrier_name,carrier_type,share,unit
146,LE_00058,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2025,input,CAR_00009,Hydrogen,fuel,1.000,kWh
150,LE_00059,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2030,input,CAR_00009,Hydrogen,fuel,1.000,kWh
154,LE_00060,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2040,input,CAR_00009,Hydrogen,fuel,1.000,kWh
158,LE_00061,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2040,input,CAR_00009,Hydrogen,fuel,1.000,kWh
162,LE_00062,TECH_00035,CO2 Hydrogenation to Methanol,PROC_00028,Methanol Synthesis,2050,input,CAR_00009,Hydrogen,fuel,1.000,kWh
141,LE_00057,TECH_00034,Cooking Oil HEFA Hydrotreatment,PROC_00027,SAF Production via Hydrotreatment of Used Cook...,2025,input,CAR_00009,Hydrogen,fuel,0.099,kWh
22,LE_00010,TECH_00009,El Fuel Cell SOFC,PROC_00004,Methane-Fueled Fuel Cell Electricity and Heat ...,2040,input,CAR_00009,Hydrogen,fuel,1.000,kWh
25,LE_00011,TECH_00009,El Fuel Cell SOFC,PROC_00004,Methane-Fueled Fuel Cell Electricity and Heat ...,2050,input,CAR_00009,Hydrogen,fuel,1.000,kWh
92,LE_00042,TECH_00025,H2 Boiler Thermal Power Plant,PROC_00019,Hydrogen-Fueled Boiler,2050,input,CAR_00009,Hydrogen,fuel,1.000,kWh
20,LE_00009,TECH_00008,H2 CCGT Electricity,PROC_00001,Methane-Fueled CCGT with CCS Electricity Gener...,2050,input,CAR_00009,Hydrogen,fuel,1.000,kWh


,direction,tech_name,process_name,linked_entities,first_year,last_year
0,input,CO2 Hydrogenation to Methanol,Methanol Synthesis,5,2025,2050
1,input,Cooking Oil HEFA Hydrotreatment,SAF Production via Hydrotreatment of Used Cook...,1,2025,2025
2,input,El Fuel Cell SOFC,Methane-Fueled Fuel Cell Electricity and Heat ...,2,2040,2050
3,input,H2 Boiler Thermal Power Plant,Hydrogen-Fueled Boiler,1,2050,2050
4,input,H2 CCGT Electricity,Methane-Fueled CCGT with CCS Electricity Gener...,1,2050,2050
5,input,H2 Fuel Cell El Building,Methane-Fueled Fuel Cell Electricity and Heat ...,2,2040,2050
6,input,H2 OCGT El,Methane-Fueled Open Cycle Gas Turbine Electric...,1,2050,2050
7,input,H2 RWGS CO,Reverse Water Gas Shift,1,2030,2030
8,input,H2 Sabatier CH4,Sabatier Reactor,4,2025,2050
9,input,HaberBoschAmmoniaSynthesis,Ammonia Synthesis (Haber–Bosch Process),1,2050,2050
